# Phase 1: AI Agent Environment & Basic LangGraph Setup
In this notebook, we are only testing that we can connect to the Gemini API and that LangGraph can maintain conversational memory. We are **not** connecting any database tools yet.

### Step 1: Load Environment and Initialize LLM

In [16]:
import os
from dotenv import load_dotenv

# Load .env file
load_dotenv('../.env')

if 'GEMINI_API_KEY' not in os.environ:
    print("WARNING: GEMINI_API_KEY not found in environment!")
else:
    print("GEMINI_API_KEY loaded successfully.")

GEMINI_API_KEY loaded successfully.


In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize the model
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [18]:
# Test a basic call
response = llm.invoke("Hello, are you ready to help sort some yarn?")
print(response.content)

[{'type': 'text', 'text': 'Hello! Absolutely, I am ready. Yarn organization is deeply satisfying!\n\nHow would you like to sort it? We can organize it by:\n\n* **Weight / Thickness** (Lace, Fingering, DK, Worsted, Chunky, etc.)\n* **Fiber Content** (Wool, Cotton, Acrylic, Alpaca, Blends)\n* **Color / Palette** \n* **Project Suitability** (Socks, Sweaters, Blankets)\n* **Brand / Line**\n* **Yardage / Skein size**\n\nTell me what yarn you have (or paste a list/description), and let me know your preferred sorting criteria!', 'extras': {'signature': 'EpgJCpUJARFNMg9rPABtPbAGJMNRD9qhyYY2CV9nwDx+3whi/eYCsNieptFziL+ZQS23luuYjUmv+VfGBwGZYHLtxv2vpVXGKPMzVosjbuA30qLFN074DnsTS6RdrtDt1tXaPOBYpnN3Akq61BuCeDTngsLc9e5yMQCuNFnWSr1nu4jBiW957+wkfa+FLUygDZdNkbPEOO2i64nxLCWIzE0ckCrPTBDdVop70pwAx2vBsT9C306eqcK+58p4RAk0nnUs3go5uF+JUpJ7KRXhtnRQ0Dv5zaPbs8tbbqFlrKTXmPnbmf0nSNdcMWf8/p4bMu2u3jMT/cwW9qgknYnC0cKvpSri6fwAjKxJn2cAvxwsq0qkV18LZVRsqZJI9WWeoO8Gqc82RHl7C2jY8z2KEfeCasx8NsoiJexxXDnBgMs8DCpIjb7dhTAGr+G210z

### Step 2: Build a Basic LangGraph (Memory Only, No Tools)
We will use `MessagesState` which automatically keeps track of a list of chat messages.

In [19]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import Annotated, TypedDict
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # appends messages to the list, rather than overwriting them.
    messages: Annotated[list, add_messages]

def call_model(state: State):
    response = llm.invoke(state["messages"])
    return {"messages": response}

# Build the graph
builder = StateGraph(State)
builder.add_node("agent", call_model)
builder.add_edge(START, "agent")
builder.add_edge("agent", END)

# Setup in-memory checkpointer for conversational memory
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

print("Basic LangGraph compiled successfully!")

Basic LangGraph compiled successfully!


### Step 3: Test Conversational Memory
We will send a message, then a follow-up, to prove the graph remembers context.

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "test_thread_1"}}

print("--- Message 1 ---")
msg1 = HumanMessage(content="My favorite color of yarn is indigo blue.")
for event in graph.stream({"messages": [msg1]}, config):
    for value in event.values():
        print(value["messages"].content)

print("\n--- Message 2 (Testing Memory) ---")
msg2 = HumanMessage(content="What is my favorite yarn color?")
for event in graph.stream({"messages": [msg2]}, config):
    for value in event.values():
        print(value["messages"].content)

---
# Phase 2: Define Tools and Schemas
Now that the environment and memory works, we will wrap our `get_matching_yarns` backend function into a tool that the AI can call, and rebuild the LangGraph to include tool routing.

In [20]:
import sys
import os
from typing import Optional

# Add the backend directory to sys.path so we can import our modules
sys.path.append(os.path.abspath('..'))

from app.db.database import SessionLocal
from app.services.filtering import get_matching_yarns
from app.schemas.schemas import YarnFilterRequest
from langchain_core.tools import tool

@tool
def filter_yarns_tool(
    price_max: Optional[float] = None,
    tenacity_min: Optional[float] = None,
    elongation_min: Optional[float] = None,
    count_dtex_min: Optional[float] = None,
    count_dtex_max: Optional[float] = None,
    shrinkage_max: Optional[float] = None,
    twist_per_metre_min: Optional[float] = None,
    twist_per_metre_max: Optional[float] = None,
    material_type: Optional[str] = None,
    supplier: Optional[str] = None,
    tensile_strength_min: Optional[float] = None,
    supplier_tenacity_min: Optional[float] = None,
    supplier_elongation_min: Optional[float] = None,
    lustre: Optional[str] = None,
    country: Optional[str] = None,
    fully_drawn_textured: Optional[str] = None,
    lead_time_max_days: Optional[int] = None,
    moq_max: Optional[float] = None
):
    """
    Finds yarns matching technical and business requirements from the database.
    Use this tool whenever the user asks to find, search, or filter yarns.
    
    Args:
        price_max: Maximum acceptable price in dollars (e.g. "cheaper than 10 dollars").
        tenacity_min: Minimum acceptable tenacity.
        elongation_min: Minimum acceptable elongation.
        count_dtex_min: Minimum count dtex (thickness).
        count_dtex_max: Maximum count dtex (thickness).
        shrinkage_max: Maximum acceptable shrinkage percentage.
        twist_per_metre_min: Minimum twist per metre (TPM).
        twist_per_metre_max: Maximum twist per metre (TPM).
        material_type: The fiber composition or type of yarn (e.g. "elastane", "cotton", "polyester").
        supplier: The name of the supplier/vendor.
        tensile_strength_min: Minimum tensile strength.
        supplier_tenacity_min: Minimum supplier tenacity.
        supplier_elongation_min: Minimum supplier elongation.
        lustre: The visual finish/shine (e.g. "bright", "semi-dull").
        country: Country of origin.
        fully_drawn_textured: Whether it is fully drawn textured (e.g. "FDY", "DTY").
        lead_time_max_days: Maximum acceptable lead time or delivery time in days (e.g. "within 4 weeks" = 28).
        moq_max: Maximum Minimum Order Quantity (MOQ) the user is willing to accept.
    """
    print(f"\n[TOOL CALLED] filter_yarns_tool(price_max={price_max}, material_type={material_type}, lead_time={lead_time_max_days}...)")
    
    # Convert kwargs to our Pydantic schema
    req = YarnFilterRequest(
        price_max=price_max,
        tenacity_min=tenacity_min,
        elongation_min=elongation_min,
        count_dtex_min=count_dtex_min,
        count_dtex_max=count_dtex_max,
        shrinkage_max=shrinkage_max,
        twist_per_metre_min=twist_per_metre_min,
        twist_per_metre_max=twist_per_metre_max,
        material_type=material_type,
        supplier=supplier,
        tensile_strength_min=tensile_strength_min,
        supplier_tenacity_min=supplier_tenacity_min,
        supplier_elongation_min=supplier_elongation_min,
        lustre=lustre,
        country=country,
        fully_drawn_textured=fully_drawn_textured,
        lead_time_max_days=lead_time_max_days,
        moq_max=moq_max
    )
    
    db = SessionLocal()
    try:
        results = get_matching_yarns(db, req)
        
        # Format results for the LLM
        if not results:
            return "No matching yarns found for these criteria."
            
        formatted = []
        for y in results[:5]:  # Limit to top 5 to save context window
            formatted.append(
                f"Material_No: {y.Material_No}, Type: {y.Type}, Price: ${y.Price}, Lead Time: {y.lt_max_days} days"
            )
        return "\n".join(formatted)
    finally:
        db.close()

tools = [filter_yarns_tool]
llm_with_tools = llm.bind_tools(tools)
print("Tool defined with ALL properties and bound to LLM successfully!")

Tool defined with ALL properties and bound to LLM successfully!


### Step 4: Add Tool Node to LangGraph
We use `ToolNode` and `tools_condition` to automatically route the agent to the tools if it decides it needs to call one.

In [21]:
from langgraph.prebuilt import ToolNode, tools_condition

tool_node = ToolNode(tools)

def call_model_with_tools(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": response}

builder_tools = StateGraph(State)
builder_tools.add_node("agent", call_model_with_tools)
builder_tools.add_node("tools", tool_node)

builder_tools.add_edge(START, "agent")

# Conditional routing: if the agent calls a tool -> route to "tools". Otherwise -> END.
builder_tools.add_conditional_edges(
    "agent",
    tools_condition,
)
builder_tools.add_edge("tools", "agent") # Loop back to agent after tool finishes

graph_with_tools = builder_tools.compile(checkpointer=MemorySaver())
print("LangGraph with Tool Routing compiled successfully!")

LangGraph with Tool Routing compiled successfully!


### Step 5: Test Agent Tool Calling
We will ask the agent a question that requires searching the database using advanced parameters.

In [23]:
config = {"configurable": {"thread_id": "tool_test_thread"}}

print("--- Query with Tools ---")
# Now the LLM can parse natural language "bright", "under 4 weeks", "less than $12" into the exact tool parameters!
msg = HumanMessage(content="Find me a elastane yarn from South Korea that is cheaper than 12 dollars and can be delivered within 4 weeks.")

for event in graph_with_tools.stream({"messages": [msg]}, config):
    for node_name, value in event.items():
        if node_name == "agent":
            # It might output a message or a tool call
            msg_obj = value["messages"]
            if hasattr(msg_obj, "tool_calls") and msg_obj.tool_calls:
                print(f"Agent decided to call tools: {msg_obj.tool_calls}")
            elif msg_obj.content:
                print(f"Agent says: {msg_obj.content}")
        elif node_name == "tools":
            print(f"Tool returned:\n{value['messages'][0].content}")

--- Query with Tools ---
Agent decided to call tools: [{'name': 'filter_yarns_tool', 'args': {'material_type': 'elastane', 'price_max': 12, 'lead_time_max_days': 28, 'country': 'South Korea'}, 'id': 'call_1393373', 'type': 'tool_call'}]

[TOOL CALLED] filter_yarns_tool(price_max=12.0, material_type=elastane, lead_time=28...)
Tool returned:
Material_No: 1000014, Type: Elastane, Price: $9.0, Lead Time: 21 days
Agent says: [{'type': 'text', 'text': 'A matching elastane yarn was found:\n\n* **Material No:** 1000014\n* **Type:** Elastane\n* **Country of Origin:** South Korea\n* **Price:** $9.00 (under $12)\n* **Lead Time:** 21 days (within 4 weeks)', 'extras': {'signature': 'EuwCCukCARFNMg88Lo9CRQvMF5mLB3A8+JiWJOx605OzZXY8DpD6T1+DtHLWL64aAPjvektMYK/fW+Lu+H4tQpK9E3iZD6CW0hbtlSpv3/glcqP9ZE0sRyA4angTsm9EhDAP12azbFTodvgFlwW9MZH++8c1E0T5jBFM3skiKiX71MCDEHNhv/H9lzU0DEGbh7tErGW8wF6TUncHnVzVBkrEjtdg3EzK2zbbXsnPVrkRcFTi/vtJo8oOBiC26snqARWn74ka99I+a1XQfxXOlIGg2QWl0CExbjLYdhZ09i06fNaUE3nhKt0kPlZ8qa1Sg